# Task 2: Historical CVaR Portfolio Optimization

This notebook implements and evaluates historical CVaR optimization for a 10-stock portfolio. It is designed for a poster-ready workflow: clean data processing, explicit optimization formulation, out-of-sample backtesting, rolling monthly re-optimization, mean-CVaR, efficient frontier diagnostics, and professional figures.

**Important project note:** `task2.ipynb` was not present in the shared workspace when this notebook was created, so this file is a complete implementation template rather than a line-by-line repair of an existing notebook.

## What Was Missing Or Incomplete

A complete project should contain the following pieces:

1. A clear historical-scenario CVaR formulation, including variables, objective, and constraints.
2. A clean chronological train/test split: 10 years for optimization and 3 years for out-of-sample evaluation.
3. Five portfolio strategies: equal-weight buy-and-hold, static CVaR, rolling monthly CVaR, static mean-CVaR, and rolling monthly mean-CVaR.
4. No look-ahead bias: rolling portfolios must use only data available before each rebalance month.
5. Professional outputs: cumulative wealth, drawdowns, allocation charts, metrics table, Sharpe ratio, and optional efficient frontier.
6. A short interpretation bridge explaining how historical data becomes CVaR scenarios and why this is not Monte Carlo sampling.

This notebook fills those gaps.

## Theory-To-Practice Bridge

Let `R` be the matrix of daily historical simple returns. Each row is one historical scenario and each column is one stock. For portfolio weights `w`, the scenario loss is:

`loss_t(w) = -R_t w`

At confidence level `alpha = 0.95`, CVaR is the average loss in the worst 5% of scenarios. The historical CVaR minimization problem is the Rockafellar-Uryasev linear program:

`minimize eta + (1 / ((1 - alpha) * T)) * sum(u_t)`

subject to:

`u_t >= -R_t w - eta` for all scenarios `t`

`u_t >= 0`, `sum(w) = 1`, and `w_i >= 0`

`eta` is the optimized VaR threshold and `u_t` captures losses beyond that threshold.

**Historical, not Monte Carlo:** this notebook does not simulate random return paths. It uses the actual historical daily return vectors as empirical scenarios with equal probability. That makes the optimization deterministic once the data window and constraints are fixed.

**Where noise enters:** the linear program itself has no random noise. The uncertainty comes from estimation error and regime mismatch: old historical returns may not represent the future distribution. Using 10 years gives more tail scenarios, but it can also include stale market regimes. The 3-year out-of-sample backtest and rolling monthly re-optimization are practical checks against this risk.

## Setup

If you run this in Colab and `yfinance` or `scipy` is missing, uncomment the install line first.

In [ ]:
# Uncomment in Colab if needed.
# %pip -q install yfinance scipy

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from scipy import sparse
from scipy.optimize import linprog

try:
    import yfinance as yf
except ImportError:
    yf = None

warnings.filterwarnings("ignore", category=FutureWarning)

plt.rcParams.update({
    "figure.figsize": (11, 6),
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.titleweight": "bold",
    "axes.titlesize": 14,
    "axes.labelsize": 11,
    "legend.frameon": False,
    "font.size": 10,
})

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

## Configuration

Replace `TICKERS` with your exact 10 stocks before producing final poster results. The default list is only a diversified example.

In [ ]:
TICKERS = ["NVDA", "AAPL", "MSFT", "AMZN", "GOOGL", "AVGO", "META", "TSLA", "WMT", "AMD"]

# For reproducible poster results, set fixed dates instead of leaving END_DATE as None.
START_DATE = "2010-01-01"
END_DATE = None

# Set LOCAL_PRICE_FILE to a CSV with a Date column and ticker columns if you already have prices.
LOCAL_PRICE_FILE = None
DATE_COLUMN = "Date"

TRAIN_YEARS = 10
TEST_YEARS = 3
USE_MOST_RECENT_WINDOW = True

CONFIDENCE_LEVEL = 0.95
TRADING_DAYS = 252
REBALANCE_FREQ = "M"

# Long-only portfolio by default. Set MAX_WEIGHT below 1.0 if you want diversification caps.
MIN_WEIGHT = 0.0
MAX_WEIGHT = 1.0

# Annual risk-free rate used only for Sharpe ratio. Keep 0.0 if you want excess return vs cash ignored.
RISK_FREE_RATE = 0.0

# Optional transaction cost for monthly rebalancing. 0 keeps the comparison clean.
TRANSACTION_COST_BPS = 0.0

EXPORT_FIGURES = True
FIGURE_DIR = Path("task2_figures")

if len(TICKERS) != 10:
    raise ValueError("This project expects exactly 10 tickers.")

## Data Loading And Processing

We use adjusted close prices when downloading from Yahoo Finance. The returns are daily simple returns, not log returns, because simple returns map naturally into portfolio wealth and buy-and-hold performance.

In [ ]:
def load_prices(tickers, start_date, end_date=None, local_price_file=None, date_column="Date"):
    """Load adjusted close prices from a local CSV or Yahoo Finance."""
    if local_price_file is not None:
        raw = pd.read_csv(local_price_file)
        if date_column not in raw.columns:
            raise ValueError(f"Local price file must contain a '{date_column}' column.")
        raw[date_column] = pd.to_datetime(raw[date_column])
        prices = raw.set_index(date_column)
        missing = [ticker for ticker in tickers if ticker not in prices.columns]
        if missing:
            raise ValueError(f"Missing ticker columns in local CSV: {missing}")
        prices = prices[tickers]
    else:
        if yf is None:
            raise ImportError("Install yfinance or provide LOCAL_PRICE_FILE.")
        data = yf.download(
            tickers,
            start=start_date,
            end=end_date,
            auto_adjust=True,
            progress=False,
            group_by="column",
        )
        if isinstance(data.columns, pd.MultiIndex):
            if "Close" in data.columns.get_level_values(0):
                prices = data["Close"]
            elif "Adj Close" in data.columns.get_level_values(0):
                prices = data["Adj Close"]
            else:
                raise ValueError("Could not find Close or Adj Close prices in downloaded data.")
        else:
            prices = data[["Close"]].rename(columns={"Close": tickers[0]})
        prices = prices.reindex(columns=tickers)

    prices = prices.sort_index()
    prices = prices[~prices.index.duplicated(keep="last")]
    prices = prices.dropna(how="all").ffill().dropna()
    prices = prices.astype(float)

    if prices.empty:
        raise ValueError("Price data is empty after cleaning.")

    return prices


def compute_returns(prices):
    returns = prices.pct_change().dropna(how="any")
    returns = returns.replace([np.inf, -np.inf], np.nan).dropna(how="any")
    if returns.empty:
        raise ValueError("Return data is empty after cleaning.")
    return returns


prices = load_prices(TICKERS, START_DATE, END_DATE, LOCAL_PRICE_FILE, DATE_COLUMN)
returns = compute_returns(prices)

coverage = pd.DataFrame({
    "first_price": prices.apply(lambda s: s.first_valid_index()),
    "last_price": prices.apply(lambda s: s.last_valid_index()),
    "missing_prices_after_cleaning": prices.isna().sum(),
})

print(f"Price observations: {len(prices):,}")
print(f"Return observations: {len(returns):,}")
print(f"Full return sample: {returns.index.min().date()} to {returns.index.max().date()}")
display(coverage)

## Exploratory Data Analysis For Poster Inputs

Before optimization, we inspect the historical return scenarios. These EDA charts help explain the input data used by the CVaR model: individual return distributions, stock co-movement, rolling volatility, risk-return position, and the main multivariate correlation structure.

In [ ]:
if EXPORT_FIGURES:
    FIGURE_DIR.mkdir(exist_ok=True)


def export_eda_figure(fig, filename):
    if EXPORT_FIGURES:
        fig.savefig(FIGURE_DIR / filename, bbox_inches="tight")


var_threshold = returns.quantile(1.0 - CONFIDENCE_LEVEL)
eda_summary = pd.DataFrame({
    "annualized_mean_return": (1.0 + returns.mean()) ** TRADING_DAYS - 1.0,
    "annualized_volatility": returns.std(ddof=1) * np.sqrt(TRADING_DAYS),
    "skewness": returns.skew(),
    "excess_kurtosis": returns.kurtosis(),
    f"daily_var_{int(CONFIDENCE_LEVEL * 100)}_loss": -var_threshold,
    f"daily_cvar_{int(CONFIDENCE_LEVEL * 100)}_loss": returns.apply(
        lambda s: -s[s <= s.quantile(1.0 - CONFIDENCE_LEVEL)].mean()
    ),
    "worst_daily_return": returns.min(),
    "best_daily_return": returns.max(),
})

display(
    eda_summary.style.format({
        "annualized_mean_return": "{:.2%}",
        "annualized_volatility": "{:.2%}",
        f"daily_var_{int(CONFIDENCE_LEVEL * 100)}_loss": "{:.2%}",
        f"daily_cvar_{int(CONFIDENCE_LEVEL * 100)}_loss": "{:.2%}",
        "worst_daily_return": "{:.2%}",
        "best_daily_return": "{:.2%}",
        "skewness": "{:.2f}",
        "excess_kurtosis": "{:.2f}",
    })
)

In [ ]:
stock_growth = (1.0 + returns).cumprod()

fig, ax = plt.subplots(figsize=(12, 6))
for ticker in returns.columns:
    ax.plot(stock_growth.index, stock_growth[ticker], linewidth=1.8, label=ticker)
ax.set_title("Historical Growth Of $1 By Stock")
ax.set_ylabel("Growth of $1")
ax.set_xlabel("Date")
ax.yaxis.set_major_formatter(mtick.StrMethodFormatter("${x:,.2f}"))
ax.legend(ncol=5, loc="upper left")
fig.tight_layout()
export_eda_figure(fig, "eda_stock_growth.png")
plt.show()

fig, axes = plt.subplots(2, 5, figsize=(15, 6), sharex=False, sharey=False)
axes = np.asarray(axes).ravel()
for ax, ticker in zip(axes, returns.columns):
    series = returns[ticker].dropna()
    q05 = series.quantile(1.0 - CONFIDENCE_LEVEL)
    ax.hist(series, bins=60, color="#4c78a8", alpha=0.82, edgecolor="white", linewidth=0.25)
    ax.axvline(0.0, color="#222222", linewidth=1.0)
    ax.axvline(q05, color="#d62728", linestyle="--", linewidth=1.2)
    ax.set_title(ticker)
    ax.xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
fig.suptitle("Daily Return Distributions With 5% Left-Tail Threshold", fontweight="bold", y=1.02)
fig.tight_layout()
export_eda_figure(fig, "eda_return_distributions.png")
plt.show()

In [ ]:
correlation = returns.corr()

fig, ax = plt.subplots(figsize=(8.5, 7.5))
image = ax.imshow(correlation, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_title("Correlation Heatmap Of Daily Returns")
ax.set_xticks(range(len(correlation.columns)))
ax.set_yticks(range(len(correlation.index)))
ax.set_xticklabels(correlation.columns, rotation=45, ha="right")
ax.set_yticklabels(correlation.index)

for i in range(len(correlation.index)):
    for j in range(len(correlation.columns)):
        value = correlation.iloc[i, j]
        text_color = "white" if abs(value) >= 0.55 else "#222222"
        ax.text(j, i, f"{value:.2f}", ha="center", va="center", color=text_color, fontsize=8)

colorbar = fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
colorbar.set_label("Correlation")
fig.tight_layout()
export_eda_figure(fig, "eda_correlation_heatmap.png")
plt.show()

In [ ]:
rolling_window = 63
rolling_volatility = returns.rolling(rolling_window).std() * np.sqrt(TRADING_DAYS)

fig, ax = plt.subplots(figsize=(12, 6))
for ticker in returns.columns:
    ax.plot(rolling_volatility.index, rolling_volatility[ticker], linewidth=1.5, label=ticker)
ax.set_title(f"Rolling {rolling_window}-Trading-Day Annualized Volatility")
ax.set_ylabel("Annualized volatility")
ax.set_xlabel("Date")
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.legend(ncol=5, loc="upper left")
fig.tight_layout()
export_eda_figure(fig, "eda_rolling_volatility.png")
plt.show()

asset_annual_return = (1.0 + returns.mean()) ** TRADING_DAYS - 1.0
asset_annual_volatility = returns.std(ddof=1) * np.sqrt(TRADING_DAYS)

fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(asset_annual_volatility, asset_annual_return, s=120, color="#2ca02c", edgecolor="white", linewidth=0.8)
for ticker in returns.columns:
    ax.annotate(ticker, (asset_annual_volatility[ticker], asset_annual_return[ticker]), xytext=(6, 4), textcoords="offset points")
ax.set_title("Individual Stock Risk And Return")
ax.set_xlabel("Annualized volatility")
ax.set_ylabel("Annualized mean return")
ax.xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
fig.tight_layout()
export_eda_figure(fig, "eda_asset_risk_return.png")
plt.show()

In [ ]:
# Principal-component view of the multivariate correlation structure.
standardized_returns = (returns - returns.mean()) / returns.std(ddof=0)
corr_matrix = standardized_returns.corr().to_numpy()
eigenvalues, eigenvectors = np.linalg.eigh(corr_matrix)
order = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[order]
eigenvectors = eigenvectors[:, order]
explained_variance = eigenvalues / eigenvalues.sum()

loadings = pd.DataFrame(
    eigenvectors[:, :2] * np.sqrt(eigenvalues[:2]),
    index=returns.columns,
    columns=["PC1", "PC2"],
)

n_components_to_plot = min(5, len(explained_variance))

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
axes[0].bar(
    range(1, n_components_to_plot + 1),
    explained_variance[:n_components_to_plot],
    color="#4c78a8",
)
axes[0].set_xticks(range(1, n_components_to_plot + 1))
axes[0].set_title("Explained Correlation Structure")
axes[0].set_xlabel("Principal component")
axes[0].set_ylabel("Share of variance")
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter(1.0))

axes[1].scatter(loadings["PC1"], loadings["PC2"], s=120, color="#f58518", edgecolor="white", linewidth=0.8)
for ticker, row in loadings.iterrows():
    axes[1].annotate(ticker, (row["PC1"], row["PC2"]), xytext=(6, 4), textcoords="offset points")
axes[1].axhline(0, color="#444444", linewidth=0.8)
axes[1].axvline(0, color="#444444", linewidth=0.8)
axes[1].set_title("Stock Loadings On First Two Components")
axes[1].set_xlabel(f"PC1 ({explained_variance[0]:.1%})")
axes[1].set_ylabel(f"PC2 ({explained_variance[1]:.1%})")

fig.suptitle("Multivariate Return Structure", fontweight="bold", y=1.02)
fig.tight_layout()
export_eda_figure(fig, "eda_multivariate_pca.png")
plt.show()

display(loadings.style.format("{:.3f}"))

## Chronological Train/Test Split

The first 10 years are used to estimate portfolio weights. The following 3 years are used only for evaluation. Yes, this 3-year period is **out-of-sample** because those returns are not used when computing the initial optimized weights.

For rolling monthly strategies, each month is optimized using the trailing 10 years available before that month begins.

In [ ]:
def make_train_test_split(returns, train_years=10, test_years=3, use_most_recent_window=True):
    required_years = train_years + test_years
    sample = returns.copy()

    if use_most_recent_window:
        analysis_start = sample.index.max() - pd.DateOffset(years=required_years)
        sample = sample.loc[sample.index >= analysis_start]

    start = sample.index.min()
    train_end = start + pd.DateOffset(years=train_years)
    test_end = train_end + pd.DateOffset(years=test_years)

    train = sample.loc[sample.index < train_end]
    test = sample.loc[(sample.index >= train_end) & (sample.index < test_end)]

    if len(train) < TRADING_DAYS * (train_years - 1):
        raise ValueError("Training window is too short. Check START_DATE and price coverage.")
    if len(test) < TRADING_DAYS * (test_years - 1):
        raise ValueError("Test window is too short. Check END_DATE and price coverage.")

    return sample, train, test


sample_returns, train_returns, test_returns = make_train_test_split(
    returns,
    train_years=TRAIN_YEARS,
    test_years=TEST_YEARS,
    use_most_recent_window=USE_MOST_RECENT_WINDOW,
)

split_summary = pd.DataFrame({
    "period": ["Training", "Out-of-sample test"],
    "start": [train_returns.index.min().date(), test_returns.index.min().date()],
    "end": [train_returns.index.max().date(), test_returns.index.max().date()],
    "trading_days": [len(train_returns), len(test_returns)],
})

display(split_summary)

## Optimization And Backtest Functions

Static portfolios below are treated as true buy-and-hold portfolios: weights are set at the start of the out-of-sample period and then allowed to drift. Rolling strategies rebalance once per month.

In [ ]:
def solve_cvar_lp(returns, alpha=0.95, target_daily_return=None, weight_bounds=(0.0, 1.0)):
    """Solve a long-only historical CVaR linear program."""
    scenario_returns = returns.to_numpy(dtype=float)
    t_count, n_assets = scenario_returns.shape

    if t_count == 0:
        raise ValueError("No return scenarios were supplied.")

    n_variables = n_assets + 1 + t_count
    eta_index = n_assets
    u_start = n_assets + 1

    c = np.zeros(n_variables)
    c[eta_index] = 1.0
    c[u_start:] = 1.0 / ((1.0 - alpha) * t_count)

    # -R_t w - eta - u_t <= 0 is equivalent to u_t >= loss_t(w) - eta.
    a_left = sparse.csr_matrix(-scenario_returns)
    a_eta = sparse.csr_matrix(-np.ones((t_count, 1)))
    a_u = -sparse.eye(t_count, format="csr")
    a_ub = sparse.hstack([a_left, a_eta, a_u], format="csr")
    b_ub = np.zeros(t_count)

    if target_daily_return is not None:
        mu = returns.mean().to_numpy(dtype=float)
        target_row_dense = np.zeros((1, n_variables))
        target_row_dense[0, :n_assets] = -mu
        target_row = sparse.csr_matrix(target_row_dense)
        a_ub = sparse.vstack([a_ub, target_row], format="csr")
        b_ub = np.concatenate([b_ub, [-target_daily_return]])

    a_eq_dense = np.zeros((1, n_variables))
    a_eq_dense[0, :n_assets] = 1.0
    a_eq = sparse.csr_matrix(a_eq_dense)
    b_eq = np.array([1.0])

    bounds = [weight_bounds] * n_assets + [(None, None)] + [(0.0, None)] * t_count

    result = linprog(
        c,
        A_ub=a_ub,
        b_ub=b_ub,
        A_eq=a_eq,
        b_eq=b_eq,
        bounds=bounds,
        method="highs",
    )

    if not result.success:
        raise RuntimeError(f"CVaR optimization failed: {result.message}")

    weights = pd.Series(result.x[:n_assets], index=returns.columns, name="weight")
    weights[np.abs(weights) < 1e-8] = 0.0
    weights = weights / weights.sum()

    eta = float(result.x[eta_index])
    objective_cvar = float(result.fun)

    return weights, {"eta_var_loss": eta, "objective_cvar_loss": objective_cvar}


def buy_and_hold_returns(asset_returns, initial_weights):
    """Return portfolio returns from initial weights without daily rebalancing."""
    weights = pd.Series(initial_weights, index=asset_returns.columns, dtype=float)
    asset_growth = (1.0 + asset_returns).cumprod()
    wealth = asset_growth.mul(weights, axis=1).sum(axis=1)
    portfolio_returns = wealth.pct_change()
    portfolio_returns.iloc[0] = wealth.iloc[0] - 1.0
    return portfolio_returns.rename("portfolio_return")


def equal_weight_target_return(window_returns):
    equal_weights = pd.Series(1.0 / window_returns.shape[1], index=window_returns.columns)
    return float((window_returns @ equal_weights).mean())


def rolling_monthly_strategy(
    all_returns,
    test_index,
    optimizer,
    train_years=10,
    transaction_cost_bps=0.0,
):
    """Optimize at the start of each test month using the trailing training window."""
    test_returns_local = all_returns.loc[test_index]
    monthly_periods = test_returns_local.index.to_period(REBALANCE_FREQ).unique()

    strategy_returns = []
    weights_by_month = []
    previous_weights = None

    for period in monthly_periods:
        month_returns = test_returns_local.loc[test_returns_local.index.to_period(REBALANCE_FREQ) == period]
        rebalance_date = month_returns.index.min()
        train_start = rebalance_date - pd.DateOffset(years=train_years)
        historical_window = all_returns.loc[(all_returns.index >= train_start) & (all_returns.index < rebalance_date)]

        if len(historical_window) < TRADING_DAYS * (train_years - 1):
            raise ValueError(f"Not enough historical data before {rebalance_date.date()} for rolling optimization.")

        weights = optimizer(historical_window)
        period_returns = buy_and_hold_returns(month_returns, weights)

        if previous_weights is not None and transaction_cost_bps > 0:
            turnover = float(np.abs(weights - previous_weights).sum())
            cost = turnover * transaction_cost_bps / 10_000.0
            period_returns.iloc[0] -= cost

        strategy_returns.append(period_returns)
        weights_by_month.append(weights.rename(rebalance_date))
        previous_weights = weights

    strategy_returns = pd.concat(strategy_returns).sort_index()
    weights_by_month = pd.DataFrame(weights_by_month)
    weights_by_month.index.name = "rebalance_date"

    return strategy_returns, weights_by_month


def historical_var_cvar(portfolio_returns, alpha=0.95):
    losses = -pd.Series(portfolio_returns).dropna().to_numpy(dtype=float)
    var_loss = float(np.quantile(losses, alpha))
    tail_losses = losses[losses >= var_loss]
    cvar_loss = float(tail_losses.mean()) if len(tail_losses) else np.nan
    return var_loss, cvar_loss


def max_drawdown(portfolio_returns):
    wealth = (1.0 + pd.Series(portfolio_returns).dropna()).cumprod()
    drawdown = wealth / wealth.cummax() - 1.0
    return float(drawdown.min())


def performance_metrics(strategy_returns, alpha=0.95, risk_free_rate=0.0):
    rows = []
    for name, series in strategy_returns.items():
        r = pd.Series(series).dropna()
        cumulative_return = float((1.0 + r).prod() - 1.0)
        annual_return = float((1.0 + cumulative_return) ** (TRADING_DAYS / len(r)) - 1.0)
        annual_volatility = float(r.std(ddof=1) * np.sqrt(TRADING_DAYS))
        sharpe = np.nan if annual_volatility == 0 else (annual_return - risk_free_rate) / annual_volatility
        var_loss, cvar_loss = historical_var_cvar(r, alpha)

        rows.append({
            "strategy": name,
            "cumulative_return": cumulative_return,
            "annual_return": annual_return,
            "annual_volatility": annual_volatility,
            "sharpe_ratio": sharpe,
            "max_drawdown": max_drawdown(r),
            f"daily_var_{int(alpha * 100)}_loss": var_loss,
            f"daily_cvar_{int(alpha * 100)}_loss": cvar_loss,
        })

    return pd.DataFrame(rows).set_index("strategy")


def format_metrics(metrics):
    percent_cols = [
        "cumulative_return",
        "annual_return",
        "annual_volatility",
        "max_drawdown",
        f"daily_var_{int(CONFIDENCE_LEVEL * 100)}_loss",
        f"daily_cvar_{int(CONFIDENCE_LEVEL * 100)}_loss",
    ]
    return metrics.style.format({col: "{:.2%}" for col in percent_cols}).format({"sharpe_ratio": "{:.2f}"})

## Run The Five Required Strategies

For mean-CVaR, the return target is set to the historical mean daily return of the equal-weight portfolio in the relevant training window. This makes the problem feasible by construction because the equal-weight portfolio satisfies the target. You can make the target more aggressive later, but feasibility will then depend on whether the target is attainable under the weight constraints.

In [ ]:
weight_bounds = (MIN_WEIGHT, MAX_WEIGHT)
equal_weights = pd.Series(1.0 / len(TICKERS), index=TICKERS, name="Equal Weight")

static_cvar_weights, static_cvar_info = solve_cvar_lp(
    train_returns,
    alpha=CONFIDENCE_LEVEL,
    target_daily_return=None,
    weight_bounds=weight_bounds,
)
static_cvar_weights.name = "CVaR 95% Static"

mean_cvar_target = equal_weight_target_return(train_returns)
static_mean_cvar_weights, static_mean_cvar_info = solve_cvar_lp(
    train_returns,
    alpha=CONFIDENCE_LEVEL,
    target_daily_return=mean_cvar_target,
    weight_bounds=weight_bounds,
)
static_mean_cvar_weights.name = "Mean-CVaR 95% Static"

def cvar_optimizer(window_returns):
    weights, _ = solve_cvar_lp(
        window_returns,
        alpha=CONFIDENCE_LEVEL,
        target_daily_return=None,
        weight_bounds=weight_bounds,
    )
    return weights


def mean_cvar_optimizer(window_returns):
    target = equal_weight_target_return(window_returns)
    weights, _ = solve_cvar_lp(
        window_returns,
        alpha=CONFIDENCE_LEVEL,
        target_daily_return=target,
        weight_bounds=weight_bounds,
    )
    return weights


strategy_returns = pd.DataFrame(index=test_returns.index)
strategy_returns["Equal Weight"] = buy_and_hold_returns(test_returns, equal_weights)
strategy_returns["CVaR 95% Static"] = buy_and_hold_returns(test_returns, static_cvar_weights)
strategy_returns["Mean-CVaR 95% Static"] = buy_and_hold_returns(test_returns, static_mean_cvar_weights)

rolling_cvar_returns, rolling_cvar_weights = rolling_monthly_strategy(
    sample_returns,
    test_returns.index,
    optimizer=cvar_optimizer,
    train_years=TRAIN_YEARS,
    transaction_cost_bps=TRANSACTION_COST_BPS,
)
strategy_returns["CVaR 95% Rolling Monthly"] = rolling_cvar_returns

rolling_mean_cvar_returns, rolling_mean_cvar_weights = rolling_monthly_strategy(
    sample_returns,
    test_returns.index,
    optimizer=mean_cvar_optimizer,
    train_years=TRAIN_YEARS,
    transaction_cost_bps=TRANSACTION_COST_BPS,
)
strategy_returns["Mean-CVaR 95% Rolling Monthly"] = rolling_mean_cvar_returns

strategy_order = [
    "Equal Weight",
    "CVaR 95% Static",
    "CVaR 95% Rolling Monthly",
    "Mean-CVaR 95% Static",
    "Mean-CVaR 95% Rolling Monthly",
]
strategy_returns = strategy_returns[strategy_order]

static_weights = pd.concat([equal_weights, static_cvar_weights, static_mean_cvar_weights], axis=1)
display(static_weights.style.format("{:.2%}"))

print(f"Mean-CVaR daily target from training equal-weight portfolio: {mean_cvar_target:.6f}")
print(f"Mean-CVaR annualized target approximation: {(1 + mean_cvar_target) ** TRADING_DAYS - 1:.2%}")

## Out-of-Sample Performance Summary

The table below evaluates realized performance over the 3-year out-of-sample period only. The optimized portfolios are judged on future data that was not used in the initial optimization.

In [ ]:
metrics = performance_metrics(strategy_returns, alpha=CONFIDENCE_LEVEL, risk_free_rate=RISK_FREE_RATE)
metrics = metrics.loc[strategy_order]
display(format_metrics(metrics))

## Poster-Ready Visualizations

In [ ]:
COLORS = {
    "Equal Weight": "#1f77b4",
    "CVaR 95% Static": "#d62728",
    "CVaR 95% Rolling Monthly": "#2ca02c",
    "Mean-CVaR 95% Static": "#9467bd",
    "Mean-CVaR 95% Rolling Monthly": "#ff7f0e",
}

if EXPORT_FIGURES:
    FIGURE_DIR.mkdir(exist_ok=True)


def save_figure(fig, filename):
    if EXPORT_FIGURES:
        fig.savefig(FIGURE_DIR / filename, bbox_inches="tight")


def plot_cumulative_wealth(strategy_returns):
    wealth = (1.0 + strategy_returns).cumprod()
    fig, ax = plt.subplots(figsize=(12, 6))
    for column in strategy_returns.columns:
        ax.plot(wealth.index, wealth[column], label=column, color=COLORS[column], linewidth=2.2)
    ax.set_title("Out-of-Sample Cumulative Wealth")
    ax.set_ylabel("Growth of $1")
    ax.set_xlabel("Date")
    ax.yaxis.set_major_formatter(mtick.StrMethodFormatter("${x:,.2f}"))
    ax.legend(ncol=2, loc="best")
    fig.tight_layout()
    save_figure(fig, "cumulative_wealth.png")
    return fig, ax


def plot_drawdowns(strategy_returns):
    wealth = (1.0 + strategy_returns).cumprod()
    drawdowns = wealth / wealth.cummax() - 1.0
    fig, ax = plt.subplots(figsize=(12, 5))
    for column in strategy_returns.columns:
        ax.plot(drawdowns.index, drawdowns[column], label=column, color=COLORS[column], linewidth=1.8)
    ax.set_title("Out-of-Sample Drawdowns")
    ax.set_ylabel("Drawdown")
    ax.set_xlabel("Date")
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
    ax.legend(ncol=2, loc="lower left")
    fig.tight_layout()
    save_figure(fig, "drawdowns.png")
    return fig, ax


def plot_static_weights(static_weights):
    fig, ax = plt.subplots(figsize=(11, 5.5))
    static_weights.T.plot(kind="bar", stacked=True, ax=ax, width=0.72, colormap="tab20")
    ax.set_title("Initial Portfolio Allocations")
    ax.set_ylabel("Weight")
    ax.set_xlabel("")
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
    ax.legend(title="Stock", bbox_to_anchor=(1.02, 1), loc="upper left")
    fig.tight_layout()
    save_figure(fig, "static_allocations.png")
    return fig, ax


def plot_rolling_weights(weights, title, filename):
    fig, ax = plt.subplots(figsize=(12, 5.5))
    weights.plot(kind="area", stacked=True, ax=ax, colormap="tab20", linewidth=0)
    ax.set_title(title)
    ax.set_ylabel("Weight")
    ax.set_xlabel("Rebalance date")
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
    ax.legend(title="Stock", bbox_to_anchor=(1.02, 1), loc="upper left")
    fig.tight_layout()
    save_figure(fig, filename)
    return fig, ax


def plot_risk_return(metrics):
    fig, ax = plt.subplots(figsize=(9, 6))
    for strategy, row in metrics.iterrows():
        ax.scatter(
            row["annual_volatility"],
            row["annual_return"],
            s=110,
            color=COLORS[strategy],
            label=strategy,
            edgecolor="white",
            linewidth=0.8,
        )
        ax.annotate(strategy, (row["annual_volatility"], row["annual_return"]), xytext=(7, 4), textcoords="offset points")
    ax.set_title("Out-of-Sample Risk And Return")
    ax.set_xlabel("Annualized volatility")
    ax.set_ylabel("Annualized return")
    ax.xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
    fig.tight_layout()
    save_figure(fig, "risk_return_scatter.png")
    return fig, ax


plot_cumulative_wealth(strategy_returns)
plot_drawdowns(strategy_returns)
plot_static_weights(static_weights)
plot_rolling_weights(rolling_cvar_weights, "Rolling Monthly CVaR Allocations", "rolling_cvar_allocations.png")
plot_rolling_weights(rolling_mean_cvar_weights, "Rolling Monthly Mean-CVaR Allocations", "rolling_mean_cvar_allocations.png")
plot_risk_return(metrics)
plt.show()

## Optional: Efficient Frontier

The efficient frontier shows the best attainable CVaR risk for different target expected returns. In a mean-CVaR project, this is useful because it visually explains the trade-off: requiring higher expected return usually forces the optimizer to accept more tail risk.

Sharpe ratio is also reported above. It is the annualized excess return divided by annualized volatility. It is not a CVaR measure, but it is familiar to audiences and useful for comparing against standard finance benchmarks.

In [ ]:
def build_mean_cvar_frontier(train_returns, points=25, alpha=0.95, weight_bounds=(0.0, 1.0)):
    mu_daily = train_returns.mean()
    target_grid = np.linspace(mu_daily.min(), mu_daily.max(), points)
    rows = []

    for target in target_grid:
        try:
            weights, info = solve_cvar_lp(
                train_returns,
                alpha=alpha,
                target_daily_return=float(target),
                weight_bounds=weight_bounds,
            )
        except RuntimeError:
            continue

        portfolio_train_returns = train_returns @ weights
        var_loss, cvar_loss = historical_var_cvar(portfolio_train_returns, alpha)
        rows.append({
            "target_daily_return": float(target),
            "target_annual_return": float((1.0 + target) ** TRADING_DAYS - 1.0),
            "realized_train_annual_return": float((1.0 + portfolio_train_returns.mean()) ** TRADING_DAYS - 1.0),
            "daily_cvar_loss": cvar_loss,
            "objective_cvar_loss": info["objective_cvar_loss"],
            **{f"w_{ticker}": weights[ticker] for ticker in train_returns.columns},
        })

    return pd.DataFrame(rows)


frontier = build_mean_cvar_frontier(
    train_returns,
    points=30,
    alpha=CONFIDENCE_LEVEL,
    weight_bounds=weight_bounds,
)

fig, ax = plt.subplots(figsize=(9.5, 6))
ax.plot(frontier["daily_cvar_loss"], frontier["target_annual_return"], color="#2f4b7c", linewidth=2.4)
ax.scatter(frontier["daily_cvar_loss"], frontier["target_annual_return"], color="#f95d6a", s=32, zorder=3)
ax.set_title("Training-Period Mean-CVaR Efficient Frontier")
ax.set_xlabel(f"Historical daily CVaR {int(CONFIDENCE_LEVEL * 100)}% loss")
ax.set_ylabel("Target annualized expected return")
ax.xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
fig.tight_layout()
save_figure(fig, "mean_cvar_frontier.png")
plt.show()

display(frontier[["target_annual_return", "daily_cvar_loss"]].style.format("{:.2%}"))

## Interpretation Checklist For The Poster

- We use historical scenario optimization, not Monte Carlo simulation.
- The EDA section summarizes the empirical return scenarios before they enter the optimizer.
- Each daily vector of 10 stock returns is one empirical scenario.
- The CVaR model is deterministic after selecting the historical window, confidence level, and constraints.
- Estimation risk remains because historical returns may not represent future market conditions.
- The 10-year training period is in-sample; the following 3-year test period is out-of-sample.
- Monthly rolling optimization uses only trailing historical data, so it avoids look-ahead bias.
- Mean-CVaR is feasible here because the return target is tied to the equal-weight portfolio, which is inside the feasible set.
- The efficient frontier explains the return-tail-risk trade-off; Sharpe ratio gives a familiar volatility-adjusted comparison.